# Liberia Economic Performance Analysis (2010–2025)

> **Practice-data disclaimer:** The dataset in this repository is illustrative/synthetic and is not verified official Liberian economic data. Use it for learning Python, SQL, and Power BI only.

## Project overview
This notebook demonstrates an end-to-end analytics workflow: data inspection and quality checks, feature engineering, exploratory analysis, visualization, and SQL analysis using SQLite.

## Questions explored
- How do GDP and annual GDP growth change over time?
- How do inflation and GDP growth move across the period?
- How do exports compare with imports, and what is the trade balance?
- How do GDP per capita, population, internet use, and electricity access change?
- Which years stand out in SQL aggregations, rankings, and risk-flag examples?

## How to run
1. Clone or download this repository.
2. Install dependencies with `pip install -r requirements.txt`.
3. Open this notebook in Jupyter from the repository root.
4. Run the cells from top to bottom.

The notebook reads `data/liberia_economic_practice.csv` and exports the analysis-ready file to `data/liberia_economic_analysis.csv` for Power BI.


In [ ]:
import pandas as pd

df = pd.read_csv("data/liberia_economic_practice.csv")

df.head(20)

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

## Step 2 — Check the data quality

### 2.1 Check missing values

In [ ]:
df.isnull().sum()

2.2 Check duplicate rows

In [ ]:
df.duplicated().sum()

### 2.3 Check the data types

In [ ]:
df.dtypes

### 2.4 Check the years

In [ ]:
df["year"].unique()

2.5 Check whether every year appears once

In [ ]:
df["year"].value_counts().sort_index()

### Step 3 — Create Your Clean Dataset

In [ ]:
df_clean = df.copy()

In [ ]:
df_clean.head()

In [ ]:
df_clean.shape

### Step 3.1 — Create a New Business Metric

## Trade Balance

## Trade Balance = Exports − Imports

In [ ]:
df_clean["trade_balance_usd"] = (
    df_clean["exports_usd"] - df_clean["imports_usd"]
)

In [ ]:
df_clean[["year", "exports_usd", "imports_usd", "trade_balance_usd"]]

### Step 4.2 — Let's visualize it 📈

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.plot(
    df_clean["year"],
    df_clean["gdp_usd"],
    marker="o"
)

plt.title("Liberia GDP Trend, 2010–2025")
plt.xlabel("Year")
plt.ylabel("GDP (USD)")
plt.grid(True)

plt.show()

### Step 4.3 — Analyze GDP Growth Rate

In [ ]:
df_clean[["year", "gdp_growth_pct"]]

In [ ]:
df_clean.loc[df_clean["gdp_growth_pct"].idxmax()]

In [ ]:
df_clean.loc[df_clean["gdp_growth_pct"].idxmax()]

### Let's test part of your hypothesis

In [ ]:
df_clean["export_growth_pct"] = df_clean["exports_usd"].pct_change() * 100

In [ ]:
df_clean[["year", "exports_usd", "export_growth_pct"]]

In [ ]:
df_clean[df_clean["year"].isin([2020, 2021])]

### Step 4.5 — Inflation Analysis

In [ ]:
df_clean[["year", "inflation_pct"]]

In [ ]:
df_clean.loc[df_clean["inflation_pct"].idxmax()]

## Step 5 — Population Growth Analysis

How has Liberia's population changed over time, and how does population growth compare with economic growth?

The annual population growth rate

In [ ]:
df_clean["population_growth_pct"] =  ( df_clean["population"].pct_change() * 100
)

In [ ]:
df_clean[["year" , "population", "population_growth_pct"]]

## Step 6- Per capita

In [ ]:
df_clean[["year", "gdp_per_capita_usd"]]

In [ ]:
df_clean.loc[df_clean["gdp_per_capita_usd"].idxmax()]

In [ ]:
df_clean.loc[df_clean["gdp_per_capita_usd"].idxmin()]

## Step 7 -Economic relationships

In [ ]:
df_clean[["gdp_growth_pct", "inflation_pct", "population_growth_pct" ,"gdp_per_capita_usd", "export_growth_pct"]].corr()

In [ ]:

df_clean[["year", "inflation_pct" , "gdp_growth_pct"]]

## Step 9-Trade Analysis

In [ ]:
df_clean[["year","exports_usd", "imports_usd","trade_balance_usd"]]

#### Trade & GDP


In [ ]:
print(df_clean.columns.tolist())


In [ ]:
df_clean[["year","gdp_growth_pct","exports_usd","imports_usd"]]

#### export growth

In [ ]:
df_clean["export_growth_pct"] = (df_clean["exports_usd"].pct_change() * 100 ) 
df_clean[["year","exports_usd","export_growth_pct"]]

## Step 10-Digital Development Indicators

In [ ]:
df_clean[["year", "internet_users_pct", "electricity_access_pct"]]

## Step 11 -Compare Economic Indicators

In [ ]:
df_clean[["year","gdp_growth_pct","inflation_pct","gdp_per_capita_usd","population_growth_pct","internet_users_pct","electricity_access_pct"]]

## Step 12: My first Data Analyst summary

In [ ]:
df_clean.describe().T

## Step 13: Move your Liberia analysis into SQL

In [ ]:
### 13.1 Create a SQLite database

In [ ]:
import sqlite3

conn = sqlite3.connect("liberia_economic.db")
cursor = conn.cursor()

df_clean.to_sql(
    "economic_data",
    conn,
    if_exists="replace",
    index=False
)

print("Database and table created successfully.")

In [ ]:
cursor.execute("""
SELECT *
FROM economic_data
LIMIT 5;
""")

results = cursor.fetchall()

for row in results:
    print(row)

In [ ]:
cursor.execute("""
SELECT year, gdp_usd, gdp_growth_pct
FROM economic_data;
""")

results = cursor.fetchall()

for row in results:
    print(row)

In [ ]:
cursor.execute("""
SELECT year, inflation_pct
FROM economic_data
WHERE inflation_pct > 10;
""")

results = cursor.fetchall()

for row in results:
    print(row)
    

In [ ]:
cursor.execute("""
SELECT year, inflation_pct
FROM economic_data
ORDER BY inflation_pct DESC;
""")

results = cursor.fetchall()

for row in results:
    print(row)

In [ ]:
### SQL_Aggregation

AVG() — Average

In [ ]:
cursor.execute("""
SELECT AVG(gdp_growth_pct) AS average_gdp_growth
FROM economic_data;
""")

result = cursor.fetchone()

print(result)

MAX() and MIN()

Highest GDP growth

In [ ]:
cursor.execute("""
SELECT MAX(gdp_growth_pct) AS highest_gdp_growth
FROM economic_data;
""")

print(cursor.fetchone()[0])

Lowest GDP growth

In [ ]:
cursor.execute("""
SELECT MIN(gdp_growth_pct) AS lowest_gdp_growth
FROM economic_data;
""")

print(cursor.fetchone()[0])

In [ ]:
 ## SUM()

Let's calculate total exports from 2010–2025.

In [ ]:
cursor.execute("""
SELECT SUM(exports_usd) AS total_exports
FROM economic_data;
""")

print(cursor.fetchone()[0])

### Multiple Aggregations in One Query

In [ ]:
cursor.execute("""
SELECT
    AVG(exports_usd) AS average_exports,
    AVG(imports_usd) AS average_imports,
    MAX(exports_usd) AS highest_exports,
    MIN(exports_usd) AS lowest_exports
FROM economic_data;
""")

result = cursor.fetchone()

print(result)

### GROUP_BY

In [ ]:
cursor.execute("""
SELECT
    CASE
        WHEN year <= 2017 THEN '2010-2017'
        ELSE '2018-2025'
    END AS period,
    AVG(gdp_growth_pct) AS average_gdp_growth
FROM economic_data
GROUP BY period;
""")

results = cursor.fetchall()

for row in results:
    print(row)

### CASE WHEN

In [ ]:
cursor.execute("""
SELECT
    year,
    exports_usd,
    imports_usd,
    CASE
        WHEN exports_usd > imports_usd THEN 'Trade Surplus'
        WHEN exports_usd < imports_usd THEN 'Trade Deficit'
        ELSE 'Balanced'
    END AS trade_status
FROM economic_data;
""")

results = cursor.fetchall()

for row in results:
    print(row)

### GROUP BY + COUNT ()

In [ ]:
cursor.execute(""" 
SELECT
   CASE 
       WHEN exports_usd > imports_usd THEN 'Trade Surplus'
       WHEN exports_usd < imports_usd THEN 'Trade Deficit'
       ELSE 'Balanced'
    END AS trade_status,
    COUNT(*) AS years
FROM economic_data
GROUP BY trade_status;""")
results = cursor.fetchall()
for row in results:
    print(row)

In [ ]:
cursor.execute("""
SELECT
  year,
  exports_usd,
  imports_usd,
  exports_usd - imports_usd AS 
  trade_balance_usd
  FROM economic_data;""")
results = cursor.fetchall()
for row in results:
    print(row)

### ROUND()

In [ ]:
cursor.execute("""
SELECT
    ROUND(AVG(exports_usd -imports_usd), 2) AS 
average_trade_balance
FROM economic_data;""")
result = cursor.fetchone()
print(result[0])

## Build an Economic KPI Summary 📊

In [ ]:
cursor.execute("""
SELECT
    COUNT(*) AS total_years,
    ROUND(AVG(gdp_growth_pct), 2) AS avg_gdp_growth,
    ROUND(AVG(inflation_pct), 2) AS avg_inflation,
    ROUND(AVG(exports_usd), 2) AS avg_exports,
    ROUND(AVG(imports_usd), 2) AS avg_imports,
    ROUND(AVG(exports_usd - imports_usd), 2) AS avg_trade_balance
FROM economic_data;
""")

result = cursor.fetchone()
print(result)

### Year-over-Year Export Growth with LAG()

In [ ]:
cursor.execute("""
SELECT
    year,
    exports_usd,
    LAG(exports_usd) OVER (ORDER BY year) AS previous_year_exports
FROM economic_data;
""")

results = cursor.fetchall()

for row in results:
    print(row)

In [ ]:
cursor.execute("""
WITH gdp_changes AS (
    SELECT
        year,
        gdp_usd,
        LAG(gdp_usd) OVER (ORDER BY year) AS previous_year_gdp
    FROM economic_data
)
SELECT
    year,
    gdp_usd,
    previous_year_gdp,
    gdp_usd - previous_year_gdp AS gdp_change_usd
FROM gdp_changes
WHERE gdp_usd < previous_year_gdp;
""")

results = cursor.fetchall()

for row in results:
    print(row)

In [ ]:
cursor.execute("""
WITH gdp_changes AS (
    SELECT
        year,
        gdp_usd,
        LAG(gdp_usd) OVER (ORDER BY year) AS previous_year_gdp
    FROM economic_data
)
SELECT
    year,
    ROUND(gdp_usd - previous_year_gdp, 2) AS gdp_change_usd
FROM gdp_changes
WHERE gdp_usd < previous_year_gdp
ORDER BY gdp_change_usd ASC
LIMIT 1;
""")

result = cursor.fetchone()
print(result)

## Create an Analyst-Ready SQL Analysis Table 📊

In [ ]:
cursor.execute("""
SELECT
    year,
    gdp_usd,
    gdp_growth_pct,
    inflation_pct,
    exports_usd,
    imports_usd,
    exports_usd - imports_usd AS trade_balance_usd,
    internet_users_pct,
    electricity_access_pct
FROM economic_data
ORDER BY year;
""")

results = cursor.fetchall()

for row in results:
    print(row) 

## Find economically difficult years

In [ ]:
cursor.execute("""
SELECT
    year,
    gdp_growth_pct,
    inflation_pct,
    exports_usd - imports_usd AS trade_balance_usd
FROM economic_data
WHERE gdp_growth_pct < 0
  AND inflation_pct > 10
ORDER BY year;
""")

results = cursor.fetchall()

for row in results:
    print(row)

Use HAVING 📊

In [ ]:
cursor.execute("""
SELECT
    CASE
        WHEN year <= 2017 THEN '2010-2017'
        ELSE '2018-2025'
    END AS period,
    ROUND(AVG(gdp_growth_pct), 2) AS avg_gdp_growth,
    ROUND(AVG(inflation_pct), 2) AS avg_inflation
FROM economic_data
GROUP BY period
HAVING AVG(gdp_growth_pct) < 3;
""")

results = cursor.fetchall()

for row in results:
    print(row)

Find the top 5 export years

In [ ]:
cursor.execute("""
SELECT
    year,
    exports_usd
FROM economic_data
ORDER BY exports_usd DESC
LIMIT 5;
""")

results = cursor.fetchall()

for row in results:
    print(row)

### Rank Years by Exports 🏆

Instead of only getting the top 5, let's make SQL assign a rank to every year.

Use RANK():

In [ ]:
cursor.execute("""
SELECT
    year,
    exports_usd,
    RANK() OVER (ORDER BY exports_usd DESC) AS export_rank
FROM economic_data
ORDER BY export_rank;
""")

results = cursor.fetchall()

for row in results:
    print(row)

Step 18 — Find the 3 lowest-growth years

Now change the query so that we rank from lowest to highest:

In [ ]:
cursor.execute("""
SELECT
    year,
    gdp_growth_pct,
    RANK() OVER (ORDER BY gdp_growth_pct ASC) AS growth_rank
FROM economic_data
ORDER BY growth_rank
LIMIT 3;
""")

results = cursor.fetchall()

for row in results:
    print(row)

Step 19 — Combine GDP Growth and Inflation

Now let's create a more realistic analyst query. We want to identify years where:

GDP growth was below the overall average
Inflation was above the overall average

In [ ]:
cursor.execute("""
SELECT
    year,
    gdp_growth_pct,
    inflation_pct
FROM economic_data
WHERE gdp_growth_pct < (
    SELECT AVG(gdp_growth_pct)
    FROM economic_data
)
AND inflation_pct > (
    SELECT AVG(inflation_pct)
    FROM economic_data
)
ORDER BY year;
""")

results = cursor.fetchall()

for row in results:
    print(row)

In [ ]:
cursor.execute("""
SELECT
    year,
    gdp_growth_pct,
    inflation_pct
FROM economic_data
WHERE gdp_growth_pct < (
    SELECT AVG(gdp_growth_pct)
    FROM economic_data
)
AND inflation_pct > (
    SELECT AVG(inflation_pct)
    FROM economic_data
)
ORDER BY year;
""")

results = cursor.fetchall()

for row in results:
    print(row)

Create an Economic Risk Flag

Now let's make SQL automatically classify every year.


In [ ]:
cursor.execute("""
SELECT
    year,
    gdp_growth_pct,
    inflation_pct,
    CASE
        WHEN gdp_growth_pct < 0 AND inflation_pct > 10
            THEN 'High Risk'
        WHEN gdp_growth_pct < 3 AND inflation_pct > 10
            THEN 'Moderate Risk'
        ELSE 'Lower Risk'
    END AS economic_risk
FROM economic_data
ORDER BY year;
""")

results = cursor.fetchall()

for row in results:
    print(row)

Absolutely. You’re ready for CTEs (Common Table Expressions) and more advanced analyst-style SQL.

The key idea is:

A CTE lets you build an intermediate result first, then query that result as if it were a temporary table.

1. Your first CTE

Let's calculate trade balance first, then analyze it.

In [ ]:
cursor.execute("""
WITH trade_analysis AS (
    SELECT
        year,
        exports_usd,
        imports_usd,
        exports_usd - imports_usd AS trade_balance_usd
    FROM economic_data
)
SELECT
    year,
    trade_balance_usd
FROM trade_analysis
ORDER BY trade_balance_usd ASC;
""")

results = cursor.fetchall()

for row in results:
    print(row)

2. CTE + ranking

Now let's rank the years by their trade deficit.

cursor.execute("""
WITH trade_analysis AS (
    SELECT
        year,
        exports_usd,
        imports_usd,
        exports_usd - imports_usd AS trade_balance_usd
    FROM economic_data
),
ranked_trade AS (
    SELECT
        year,
        trade_balance_usd,
        RANK() OVER (
            ORDER BY trade_balance_usd ASC
        ) AS deficit_rank
    FROM trade_analysis
)
SELECT
    year,
    trade_balance_usd,
    deficit_rank
FROM ranked_trade
WHERE deficit_rank <= 5
ORDER BY deficit_rank;
""")

results = cursor.fetchall()

for row in results:
    print(row)

Now we're using two CTEs:

economic_data
      ↓
trade_analysis
      ↓
ranked_trade
      ↓
Top 5 deficits

This is much cleaner than putting everything into one enormous SQL query.

Final Advanced SQL — PARTITION BY

In [ ]:
cursor.execute("""
WITH period_data AS (
    SELECT
        year,
        gdp_growth_pct,
        CASE
            WHEN year <= 2017 THEN '2010-2017'
            ELSE '2018-2025'
        END AS period
    FROM economic_data
)
SELECT
    year,
    period,
    gdp_growth_pct,
    RANK() OVER (
        PARTITION BY period
        ORDER BY gdp_growth_pct DESC
    ) AS growth_rank
FROM period_data
ORDER BY period, growth_rank;
""")

results = cursor.fetchall()

for row in results:
    print(row)

Step 1 — Prepare the dataset for Power BI

In [ ]:
df_clean.to_csv(
    "data/liberia_economic_analysis.csv",
    index=False
)

print("Final Power BI dataset created successfully.")

In [ ]:
pd.read_csv("data/liberia_economic_analysis.csv").head()

In [ ]:
df_clean.to_csv(
    "data/liberia_economic_analysis.csv",
    index=False
)

In [ ]:
import os
os.path.abspath("data/liberia_economic_analysis.csv")

## Reproducibility and limitations

This is a learning project based on an illustrative practice dataset. Its findings should not be presented as official facts about Liberia. The dataset has only 16 annual observations, and descriptive relationships or correlations do not establish causation.

The SQLite database is generated locally when the SQL section runs; it is intentionally not tracked in Git.


In [ ]:
conn.close()